In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import MiniBatchKMeans
from scipy.cluster.hierarchy import leaves_list, linkage
import decoupler as dc
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

In [ ]:
BASE = Path.cwd()
OUT_DIR = BASE / "results" / "04_spatial_niche_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

annotated = ad.read_zarr(BASE / "data" / "integrated_mouse_annotated.zarr")
data = annotated.obs[["cell_id", "condition", "slide", "radiation", "time", "section", "cell_subtype"]].astype(str).reset_index(drop=True)
data[["x", "y"]] = np.asarray(annotated.obsm["spatial"])
SECTIONS = list(annotated.obs["section"].cat.categories)
del annotated
data = data[data["cell_subtype"] != "Unassigned"].reset_index()

In [ ]:
def get_windows(job,n_neighbors):
    tissue_name,indices = job

    tissue = tissue_group.get_group(tissue_name)
    to_fit = tissue.loc[indices][[X,Y]].values

    fit = NearestNeighbors(n_neighbors=n_neighbors).fit(tissue[[X,Y]].values)
    m = fit.kneighbors(to_fit)

    args = m[0].argsort(axis = 1)
    add = np.arange(m[1].shape[0])*m[1].shape[1]
    sorted_indices = m[1].flatten()[args+add[:,None]]

    neighbors = tissue.index.values[sorted_indices]

    return neighbors.astype(np.int32)

In [ ]:
ks = [10,20]
X = 'x'
Y = 'y'
reg = 'section'
cluster_col = 'cell_subtype'

n_neighbors = max(ks)
cells = data

cells = pd.concat([cells,pd.get_dummies(cells[cluster_col])],axis=1)

sum_cols = cells[cluster_col].unique()
values = cells[sum_cols].values

tissue_group = cells[[X,Y,reg]].groupby(reg)
exps = list(cells[reg].unique())
tissue_chunks = [(t,indices) for t,indices in tissue_group.groups.items()]
tissues = [get_windows(job,n_neighbors) for job in tissue_chunks]

out_dict = {}
for k in ks:
    for neighbors,job in zip(tissues,tissue_chunks):

        chunk = np.arange(len(neighbors))
        tissue_name,indices = job
        window = values[neighbors[chunk,:k].flatten()].reshape(len(chunk),k,len(sum_cols)).sum(axis = 1)
        out_dict[(tissue_name,k)] = (window.astype(np.float16),indices)

windows = {}
for k in ks:

    window = pd.concat([pd.DataFrame(out_dict[(exp,k)][0],index = out_dict[(exp,k)][1].astype(int),columns = sum_cols) for exp in exps],axis=0)
    window = window.loc[cells.index.values]
    windows[k] = window

In [ ]:
k = 10
n_neighborhoods = 15
neighborhood_name = "neighborhood"+str(k)
k_centroids = {}

windows2 = windows[10]

km = MiniBatchKMeans(n_clusters = n_neighborhoods,random_state=0,batch_size=100,n_init=3)

labelskm = km.fit_predict(windows2[sum_cols].values)
k_centroids[k] = km.cluster_centers_
cells['neighborhood10'] = labelskm
cells[neighborhood_name] = cells[neighborhood_name].astype('category')

In [ ]:
k_to_plot = 10
niche_clusters = (k_centroids[k_to_plot])
tissue_avgs = values.mean(axis = 0)
fc = np.log2(((niche_clusters+tissue_avgs)/(niche_clusters+tissue_avgs).sum(axis = 1, keepdims = True))/tissue_avgs)
fc = pd.DataFrame(fc,columns = sum_cols)
fc.round(3).to_csv(OUT_DIR / "cn_celltype_enrichment.csv")

M = fc.iloc[leaves_list(linkage(fc.values, "average")), leaves_list(linkage(fc.values.T, "average"))]
M.index = [f"CN{c}" for c in M.index]
fig, ax = plt.subplots(figsize=(fs.COL2 / 2, 80 * fs.MM), layout="constrained")
im = ax.imshow(M, cmap="PuOr_r", vmin=-3, vmax=3, aspect="auto", interpolation="none")
ax.set_yticks(range(len(M)), M.index)
ax.set_xticks(range(M.shape[1]), M.columns, rotation=90)
fig.colorbar(im, ax=ax, shrink=0.8, label="log$_2$ enrichment")
fs.save(fig, OUT_DIR / "cn_celltype_enrichment")
plt.show()

In [ ]:
obs, group, stack, groups, width, height = cells, "neighborhood10", cluster_col, list(range(n_neighborhoods)), (n_neighborhoods * 3.5 + 50) * fs.MM, 55 * fs.MM
ct = pd.crosstab(obs[group], obs[stack]).reindex(index=groups, fill_value=0)
ct = ct.loc[:, ct.sum() > 0]
frac = ct.div(ct.sum(axis=1), axis=0) * 100
colors = [fs.CELL_COLORS.get(c, fs.LIGHTGREY) for c in ct.columns]
xs = np.arange(len(ct))
fig, ax = plt.subplots(figsize=(width, height), layout="constrained")
bottom = np.zeros(len(frac))
for c, color in zip(frac.columns, colors):
    ax.bar(xs, frac[c], bottom=bottom, width=0.8, color=color, edgecolor="white", linewidth=0.25, label=c)
    bottom += frac[c].to_numpy(dtype=float)
ax.set_xticks(xs, [f"CN{g}" for g in frac.index], rotation=90)
ax.set_ylabel("Proportion of cells (%)")
ax.set_ylim(0, 100)
handles, names = ax.get_legend_handles_labels()
fig.legend(handles[::-1], names[::-1], loc="outside right center", alignment="left")
fs.save(fig, OUT_DIR / "cn_composition")
ct.to_csv(OUT_DIR / "cn_composition_counts.csv")
plt.show()

In [ ]:
def fit_roi(ax, roi_cells):
    half = max(roi_cells[X].max() - roi_cells[X].min(), roi_cells[Y].max() - roi_cells[Y].min()) / 2
    cx, cy = (roi_cells[X].max() + roi_cells[X].min()) / 2, (roi_cells[Y].max() + roi_cells[Y].min()) / 2
    ax.set(xlim=(cx - half, cx + half), ylim=(cy + half, cy - half), aspect="equal")
    ax.axis("off")

cn_palette = dict(zip(cells["neighborhood10"].cat.categories, sns.color_palette("tab20", n_neighborhoods).as_hex()))
legend = [plt.Line2D([], [], marker="o", ls="", ms=2, color=color, label=label) for label, color in cn_palette.items()]
(OUT_DIR / "single" / "cn_map").mkdir(parents=True, exist_ok=True)
for i, section in enumerate(SECTIONS, start=1):
    roi_cells = cells[cells[reg] == section]
    fig, ax = plt.subplots(figsize=(55 * fs.MM, 42 * fs.MM), layout="constrained")
    ax.scatter(roi_cells[X], roi_cells[Y], c=[cn_palette[v] for v in roi_cells["neighborhood10"]], s=0.2, linewidths=0, rasterized=True)
    fit_roi(ax, roi_cells)
    ax.set_title(section, fontsize=6)
    fig.legend(handles=legend, loc="outside right center", title="CN", fontsize=5, title_fontsize=6, frameon=False)
    fs.save(fig, OUT_DIR / "single" / "cn_map" / f"cn_map_ROI{i}")
    plt.close(fig)

g = sns.lmplot(data = cells,x = X,y=Y,hue = 'neighborhood10',palette = 'tab20',height = 1.6,col = reg,col_order = SECTIONS,col_wrap = 7,fit_reg = False,
               scatter_kws = {'s':0.2,'linewidths':0,'rasterized':True},facet_kws = {'sharex':False,'sharey':False})
for ax, section in zip(g.axes.flat, SECTIONS):
    fit_roi(ax, cells[cells[reg] == section])
g.set_titles("{col_name}", size=6)
sns.move_legend(g, "center right", bbox_to_anchor=(1, 0.5), markerscale=6, title="CN", fontsize=6)
fs.save(g.figure, OUT_DIR / "cn_map")
plt.show()

In [ ]:
CN_INFO = {
    0: ("Enterocyte-rich", "Villus"),
    12: ("Enterocyte/goblet cell-rich", "Villus"),
    3: ("Enterocyte/T cell-rich", "Villus"),
    13: ("Villus SMC/eosinophil-rich", "Villus"),
    5: ("Plasma cell-rich", "Villus"),
    14: ("Vascular EC-rich", "Villus"),
    10: ("Villus fibroblast/telocyte-rich", "Villus"),
    8: ("Progenitor cell/enterocyte-rich", "Crypt"),
    6: ("Progenitor cell-rich", "Crypt"),
    7: ("Paneth cell-rich", "Crypt"),
    9: ("Adamdec1+ fibroblast-rich", "Crypt"),
    2: ("Macrophage/neutrophil-rich", "Crypt"),
    11: ("Trophocyte-rich", "Muscularis"),
    4: ("Muscularis SMC-rich", "Muscularis"),
    1: ("ILC-rich", "Lymphoid aggregate"),
}
LAYERS = ["Villus", "Crypt", "Muscularis", "Lymphoid aggregate"]
cn_names = {cn: f"CN{cn} {name}" for cn, (name, _) in CN_INFO.items()}

cells["cn"] = cells["neighborhood10"].astype(int)
cells["cn_name"] = pd.Categorical(cells["cn"].map(cn_names), categories=cn_names.values())
cells["layer"] = pd.Categorical(cells["cn"].map({cn: layer for cn, (_, layer) in CN_INFO.items()}), categories=LAYERS)
cells[["index", "cell_id", "condition", "slide", "radiation", "time", reg, "cn", "cn_name", "layer"]].to_parquet(BASE / "data" / "cn_schurch.parquet")
pd.DataFrame([(cn, name, layer, (cells["cn"] == cn).sum()) for cn, (name, layer) in CN_INFO.items()], columns=["cn", "name", "layer", "n_cells"]).to_csv(OUT_DIR / "cn_info.csv", index=False)

CN_COLORS = dict(zip(cn_names.values(), [
    "#fdd0a2", "#fdae6b", "#fd8d3c", "#d94801", "#e7ba52", "#bd9e39", "#8c6d31",
    "#9ecae1", "#3182bd", "#08306b", "#e7298a", "#de2d26",
    "#a1d99b", "#238b45",
    "#636363",
]))

In [ ]:
where = pd.crosstab(cells[cluster_col], cells["cn_name"], normalize="index")
enr = fc.rename(index=lambda c: f"CN{c} {CN_INFO[c][0]}").T
cnames = list(cells["cn_name"].cat.categories)
markers = {layer: [n for n in cnames if CN_INFO[int(n.split()[0][2:])][1] == layer] for layer in LAYERS}
rows = sorted(where.index, key=lambda t: (cnames.index(where.loc[t].idxmax()), -where.loc[t].max()))
holder = ad.AnnData(np.zeros((len(rows), len(cnames)), dtype=np.float32),
                    obs=pd.DataFrame({"cell_type": pd.Categorical(rows, categories=rows)}, index=rows), var=pd.DataFrame(index=cnames))
width, height = fs.COL2, 95 * fs.MM
dp = sc.pl.DotPlot(holder, markers, groupby="cell_type",
                   dot_color_df=enr.loc[rows, cnames], dot_size_df=where.loc[rows, cnames],
                   vmin=-3, vmax=3, vcenter=0,
                   figsize=(width, height))
dp.swap_axes()
dp.style(cmap="PuOr_r", dot_min=0, dot_max=1, largest_dot=40, dot_edge_lw=0.2)
dp.legend(size_title="% cells", colorbar_title="log$_2$" + chr(10) + "enrichment")
dp.make_figure()
dp.get_axes()["mainplot_ax"].set_ylabel("")
fs.save(dp.fig, OUT_DIR / "celltype_by_cn_dotplot")
where.loc[rows, cnames].round(4).to_csv(OUT_DIR / "celltype_by_cn_fraction.csv")
enr.loc[rows, cnames].round(3).to_csv(OUT_DIR / "celltype_by_cn_log2enrichment.csv")
plt.show()

In [ ]:
def roi_maps(column, colors, path, title):
    fig, axes = plt.subplots(2, 7, figsize=(fs.COL2, (54 if len(colors) > 6 else 58) * fs.MM))
    for ax, section in zip(axes.flat, SECTIONS):
        roi_cells = cells[cells[reg] == section]
        ax.scatter(roi_cells[X], roi_cells[Y], c=roi_cells[column].map(colors).values, s=0.05, linewidths=0, rasterized=True)
        fit_roi(ax, roi_cells)
        ax.set_title(section, fontsize=5)
    legend = [plt.Line2D([], [], marker="o", ls="", ms=3, color=color, label=label) for label, color in colors.items()]
    fig.legend(handles=legend, loc="center right", frameon=False, fontsize=5, title=title, title_fontsize=6)
    fig.subplots_adjust(left=0.005, right=0.80 if len(colors) > 6 else 0.87, top=0.95, bottom=0.02, wspace=0.03, hspace=0.15)
    fs.save(fig, path)
    plt.show()

roi_maps("cn_name", CN_COLORS, OUT_DIR / "cn_layer_map", "CN")
roi_maps("layer", fs.LAYER_COLORS, OUT_DIR / "layer_map", "Layer")

In [ ]:
layer_fraction = pd.crosstab(cells[reg], cells["layer"], normalize="index").reindex(SECTIONS)
layer_fraction.round(4).to_csv(OUT_DIR / "layer_fraction_by_roi.csv")

In [ ]:
GROUPS = ["C-ion", "γ-ray"]
ROWS = [[section for section in SECTIONS if section.startswith(radiation)] for radiation in GROUPS]
TICK_LABELS = {section: section.split(" ", 1)[1].replace(" (", "\n(") for section in SECTIONS}

TABLES = {}
for layer in ["Villus", "Crypt"]:
    obs = cells[cells["layer"] == layer]
    group, stack, groups = reg, "cn_name", SECTIONS
    width, height = 0.6 * fs.COL2 + 20 * fs.MM, 75 * fs.MM
    colors = {k: v for k, v in CN_COLORS.items() if k in set(obs[stack])}
    ct = pd.crosstab(obs[group], obs[stack]).reindex(index=groups, fill_value=0)
    ct = ct[list(colors)]
    frac = ct.div(ct.sum(axis=1), axis=0) * 100
    cols = list(colors.values())
    fig, axes = plt.subplots(len(ROWS), 1, figsize=(width, height), layout="constrained", squeeze=False, sharey="col")
    for r, row in enumerate(ROWS):
        ax = axes[r, 0]
        table = frac.loc[row]
        xs = np.arange(len(table))
        bottom = np.zeros(len(table))
        for c, color in zip(table.columns, cols):
            ax.bar(xs, table[c], bottom=bottom, width=0.75, color=color, edgecolor="white", linewidth=0.25, label=c)
            bottom += table[c].to_numpy(dtype=float)
        ax.set_xticks(xs, [TICK_LABELS[section] for section in table.index])
        ax.set_title(GROUPS[r])
        ax.set_ylabel(f"{layer}: proportion of cells (%)")
        ax.set_ylim(0, 100)
    handles, names = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles[::-1], names[::-1], loc="outside right center", alignment="left")
    fs.save(fig, OUT_DIR / f"layer_cn_composition_by_roi_{layer.lower()}")
    TABLES[layer] = ct
    plt.show()
pd.concat(TABLES, names=["layer"]).to_csv(OUT_DIR / "layer_cn_composition_by_roi_counts.csv")

In [ ]:
def composition(obs, group, stack, groups, title=None, width=fs.COL2, height=55 * fs.MM, panels=("count", "fraction"),
                rows=None, row_labels=None, tick_label=None):
    ct = pd.crosstab(obs[group], obs[stack]).reindex(index=groups, fill_value=0)
    ct = ct.loc[:, ct.sum() > 0]
    frac = ct.div(ct.sum(axis=1), axis=0) * 100
    spec = {"count": (ct, "Number of cells"), "fraction": (frac, "Proportion of cells (%)")}
    rows = rows or [groups]
    fig, axes = plt.subplots(len(rows), len(panels), figsize=(width, height), layout="constrained", squeeze=False,
                             sharey="col")
    for r, row in enumerate(rows):
        for ax, panel in zip(axes[r], panels):
            table, ylabel = spec[panel]
            table = table.loc[row]
            xs = np.arange(len(table))
            bottom = np.zeros(len(table))
            for c, color in zip(table.columns, fs.palette(table.columns)):
                ax.bar(xs, table[c], bottom=bottom, width=0.75, color=color, edgecolor="white", linewidth=0.25, label=c)
                bottom += table[c].to_numpy(dtype=float)
            if tick_label is None:
                ax.set_xticks(xs, [fs.label(g) for g in table.index], rotation=45, ha="right", rotation_mode="anchor")
            else:
                ax.set_xticks(xs, [tick_label(g) for g in table.index])
            if row_labels is not None:
                ax.set_title(row_labels[r])
            ax.set_ylabel(ylabel)
            if panel == "count":
                ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))
            else:
                ax.set_ylim(0, 100)
    handles, names = axes[0, -1].get_legend_handles_labels()
    fig.legend(handles[::-1], names[::-1], title=title, loc="outside right center", alignment="left")
    return fig, ct

CRYPT_CN = [8, 6, 7, 9, 2]
crypt_composition = {}
for cn in CRYPT_CN:
    fig, ct = composition(cells[cells["cn"] == cn], reg, cluster_col, SECTIONS, width=0.6 * fs.COL2, height=75 * fs.MM,
                             panels=("fraction",), rows=ROWS, row_labels=GROUPS, tick_label=TICK_LABELS.get)
    fig.suptitle(f"CN{cn} {CN_INFO[cn][0]}", fontsize=7)
    fs.save(fig, OUT_DIR / f"crypt_cn_composition_CN{cn}")
    plt.show()
    crypt_composition[f"CN{cn}"] = ct
pd.concat(crypt_composition, names=["cn"]).to_csv(OUT_DIR / "crypt_cn_composition_counts.csv")

In [ ]:
adata = ad.read_zarr(BASE / "data" / "integrated_mouse_annotated.zarr")
cn_all = np.full(adata.n_obs, -1)
cn_all[cells["index"].values] = cells["cn"].values
adata.obs["cn"] = cn_all
adata.obs["niche"] = [f"CN{c}" for c in cn_all]
(OUT_DIR / "volcano").mkdir(exist_ok=True)

def volcano(results_df, xlabel, path, title):
    sig = results_df[(results_df["padj"] < 0.05) & (results_df["log2FoldChange"].abs() > 0.5)]
    top_genes = list(sig.nsmallest(12, "padj").index)
    fig, ax = plt.subplots(figsize=(55 * fs.MM, 70 * fs.MM), layout="constrained")
    dc.pl.volcano(results_df.dropna(subset=["padj"]), x="log2FoldChange", y="padj",
                  top=top_genes,
                  color_pos=fs.UP, color_neg=fs.DOWN, color_null=fs.LIGHTGREY,
                  kw_scatter={"s": 4, "lw": 0, "rasterized": True}, ax=ax)
    for line in ax.lines:
        line.set(color=fs.GREY, lw=0.4, ls=(0, (2, 2)))
    for text in ax.texts:
        text.set(fontsize=5, fontstyle="italic")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("−log$_{10}$(adjusted $\\it{P}$ value)")
    ax.set_title(title, fontsize=6)
    fs.save(fig, path)
    plt.close(fig)

In [ ]:
def deseq_gamma_vs_cion(sub, groups_col):
    pdata = dc.pp.pseudobulk(
        adata=sub,
        sample_col="section",
        groups_col=groups_col,
        mode="sum",
        layer="counts",
    )
    dc.pp.filter_samples(pdata, min_cells=10, min_counts=1000)
    tcells = pdata.copy()
    dc.pp.filter_by_expr(
        adata=tcells,
        group="radiation",
        min_count=10,
        min_total_count=15,
        large_n=10,
        min_prop=0.7,
    )
    dc.pp.filter_by_prop(
        adata=tcells,
        min_prop=0.05,
        min_smpls=2,
    )

    inference = DefaultInference(n_cpus=8)
    dds = DeseqDataSet(
        adata=tcells,
        design="~radiation",
        refit_cooks=True,
        inference=inference,
        quiet=True,
    )

    dds.deseq2()

    stat_res = DeseqStats(dds, contrast=["radiation", "γ-ray", "C-ion"], inference=inference, quiet=True)

    stat_res.summary()

    return stat_res.results_df, tcells

de_results = []
for cn in CRYPT_CN:
    for time in ["48 h", "3.5 d"]:
        sub = adata[(adata.obs["cn"] == cn).values & (adata.obs["time"] == time).values]
        results_df, tcells = deseq_gamma_vs_cion(sub, "niche")
        de_results.append(results_df.assign(cn=f"CN{cn}", cn_name=CN_INFO[cn][0], time=time, n_genes=tcells.n_vars))
        if time == "3.5 d":
            volcano(results_df, f"log$_2$ fold change ({fs.label('gamma')} / {fs.label('Cion')})",
                    OUT_DIR / "volcano" / f"CN{cn}_{time.replace(' ', '')}", f"CN{cn} {CN_INFO[cn][0]}, {time}")

de_results = pd.concat(de_results).rename_axis("gene").reset_index()
de_results.to_csv(OUT_DIR / "deseq2_niche_gamma_vs_cion.csv", index=False)
sig = de_results[(de_results["padj"] < 0.05) & (de_results["log2FoldChange"].abs() > 0.5)].sort_values(["cn", "time", "padj"])

In [ ]:
counts = adata.layers["counts"].tocsr()
lib_size = np.asarray(counts.sum(axis=1)).ravel()
gene_sources = []
for (cn, time), sig_genes in sig.groupby(["cn", "time"], sort=False):
    in_niche = ((adata.obs["niche"] == cn) & (adata.obs["time"] == time)).values
    types = adata.obs["cell_subtype"].astype(str).values[in_niche]
    n_cells = pd.Series(types).value_counts()
    total = pd.Series(lib_size[in_niche]).groupby(types).sum()
    niche_counts = counts[in_niche]
    for gene in sig_genes["gene"]:
        cpm = pd.Series(niche_counts[:, adata.var_names.get_loc(gene)].toarray().ravel()).groupby(types).sum() / total * 1e6
        top = cpm[n_cells[n_cells >= 30].index].sort_values(ascending=False).head(3)
        gene_sources.append({"cn": cn, "time": time, "gene": gene, "top_cell_types (CPM)": ", ".join(f"{celltype} {cpm_value:.0f}" for celltype, cpm_value in top.items())})
gene_sources = pd.DataFrame(gene_sources)
gene_sources.to_csv(OUT_DIR / "deseq2_niche_gene_source_cpm.csv", index=False)

In [ ]:
de_troph = []
for time in ["48 h", "3.5 d"]:
    sub = adata[(adata.obs["cell_subtype"] == "Trophocyte").values & (adata.obs["cn"] == 11).values & (adata.obs["time"] == time).values]
    results_df, tcells = deseq_gamma_vs_cion(sub, "cell_subtype")
    de_troph.append(results_df.assign(cn="CN11", cell="Trophocyte", time=time, n_cells=",".join(map(str, tcells.obs["psbulk_cells"].astype(int))), n_genes=tcells.n_vars))
    if time == "3.5 d":
        volcano(results_df, f"log$_2$ fold change ({fs.label('gamma')} / {fs.label('Cion')})",
                OUT_DIR / "volcano" / f"CN11_Trophocyte_{time.replace(' ', '')}", f"Trophocyte in CN11, {time}")

de_troph = pd.concat(de_troph).rename_axis("gene").reset_index()
de_troph.to_csv(OUT_DIR / "deseq2_trophocyte_CN11_gamma_vs_cion.csv", index=False)

In [ ]:
def cluster_dotplot(adata, markers, cluster, label=None, block=None, height=100 * fs.MM, width=fs.COL2, brackets=True,
                    dot_max=1, legend_width=1.5):
    obs = adata.obs
    names = list(markers)
    genes = {ct: [g for g in gs if g in adata.var_names] for ct, gs in markers.items()}
    genes = {ct: gs for ct, gs in genes.items() if gs}
    flat = [g for gs in genes.values() for g in gs]

    table = pd.DataFrame({"cluster": obs[cluster].astype(str)})
    table["block"] = obs[block].astype(str) if block is not None else ""
    table = table.drop_duplicates("cluster").set_index("cluster", drop=False).rename_axis(None)
    table["rank"] = table["block"].map(lambda b: names.index(b) if b in names else len(names))
    table["prefix"] = table["cluster"].str.replace(r"\d+$", "", regex=True)
    table["num"] = pd.to_numeric(table["cluster"].str.extract(r"(\d+)$")[0], errors="coerce")
    if label is None:
        x = adata[:, flat].layers["log1p_norm"]
        x = x.toarray() if hasattr(x, "toarray") else np.asarray(x)
        mean = pd.DataFrame(x, columns=flat).groupby(obs[cluster].astype(str).to_numpy()).mean()
        scaled = (mean - mean.min()) / (mean.max() - mean.min())
        score = pd.DataFrame({ct: scaled[gs].mean(axis=1) for ct, gs in genes.items()})
        table["rank"] = score.loc[table.index].to_numpy().argmax(axis=1)
        blocks = list(genes.values())
        table["top_gene"] = [flat.index(scaled.loc[c, blocks[r]].idxmax()) for c, r in zip(table.index, table["rank"])]
        table = table.sort_values(["rank", "top_gene", "prefix", "num", "cluster"])
    else:
        table = table.sort_values(["rank", "prefix", "num", "cluster"])
    rows = list(table["cluster"])

    key = "_cluster_label"
    obs[key] = pd.Categorical(obs[cluster].astype(str).map(dict(zip(table["cluster"], rows))), categories=rows)
    try:
        dp = sc.pl.dotplot(adata, genes if brackets else flat, groupby=key, layer="log1p_norm", standard_scale="var",
                           dot_min=0, dot_max=dot_max,
                           figsize=(width, height), show=False, return_fig=True)
        dp.style(largest_dot=40, dot_edge_lw=0.2)
        dp.legend(width=legend_width, size_title="% cells", colorbar_title="Scaled\nexpression")
        dp.make_figure()
    finally:
        del obs[key]

    ax = dp.get_axes()["mainplot_ax"]
    for t in ax.get_xticklabels():
        t.set_fontstyle("italic")
    ax.set_ylabel("Cluster" if label is None else "Cell type")
    return dp

GENES = ["Mmp12", "Mmp13"]
sub = adata[(adata.obs["cell_subtype"].astype(str) != "Unassigned").values, GENES].copy()
font_8pt = {k: 8 for k in ["font.size", "axes.titlesize", "axes.labelsize", "xtick.labelsize", "ytick.labelsize", "legend.fontsize", "legend.title_fontsize"]}
with plt.rc_context(font_8pt):
    dp = cluster_dotplot(sub, {"All cells": GENES}, "cell_subtype", label="cell_subtype", height=110 * fs.MM, width=80 * fs.MM,
                            brackets=False,
                            dot_max=0.5)
dp.get_axes()["mainplot_ax"].set_ylabel("")
fs.save(dp.fig, OUT_DIR / "mmp_dotplot")
plt.show()

celltypes = sub.obs["cell_subtype"].astype(str).values
mmp_counts = pd.DataFrame(sub.layers["counts"].toarray(), columns=GENES)
share = (mmp_counts.groupby(celltypes).sum() / mmp_counts.sum() * 100).add_suffix(" (% of counts)")
share["cells (% of all)"] = pd.Series(celltypes).value_counts(normalize=True) * 100
share = share.sort_values("Mmp12 (% of counts)", ascending=False).round(1)
share.to_csv(OUT_DIR / "mmp_share_by_celltype.csv")
share.head(8)

In [ ]:
from sklearn.metrics import adjusted_rand_score

CONDITIONS = [condition for condition in fs.ROI_COLORS if condition.startswith(("C-ion_", "gamma-ray_"))]
CONFIGS = [(window_size, n_cn) for window_size in [10, 20] for n_cn in [10, 15, 20]]
for window_size, n_cn in CONFIGS:
    cells[f"neighborhood{window_size}_{n_cn}"] = MiniBatchKMeans(n_clusters=n_cn, random_state=0, batch_size=100, n_init=3).fit_predict(windows[window_size][sum_cols].values)
config_cols = [f"neighborhood{window_size}_{n_cn}" for window_size, n_cn in CONFIGS]
ari = pd.DataFrame({name: [adjusted_rand_score(cells["neighborhood10"], cells[name])] for name in config_cols}, index=["ARI vs main (k10, n15)"]).round(2)
ari.to_csv(OUT_DIR / "cn_sensitivity_ari.csv")

robust = []
for (window_size, n_cn), name in zip(CONFIGS, config_cols):
    cn_composition = pd.crosstab(cells[name], cells[cluster_col], normalize="index")
    cn_freq = pd.crosstab(cells[reg], cells[name], normalize="index")
    cond_freq = cn_freq.groupby(cells.groupby(reg)["condition"].first()).mean()
    for celltype in ["Progenitor cell", "Adamdec1+ fibroblast", "Macrophage"]:
        top_cn = cn_composition[celltype].idxmax()
        robust.append({"neighbors": window_size, "CNs": n_cn, "cell_type": celltype, "CN": f"CN{top_cn}", "fraction_of_CN": round(cn_composition.loc[top_cn, celltype], 2),
                       **cond_freq[top_cn].reindex(CONDITIONS).round(3).to_dict()})
robust = pd.DataFrame(robust)
robust.to_csv(OUT_DIR / "cn_sensitivity_main_niches.csv", index=False)
display(ari)
robust